In [10]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
from unidecode import unidecode
import pyreadstat
from pyspark.sql import SparkSession, functions as F

In [30]:
candidatos_data = ["/home/gus/work/data","/home/jovyan/work/data","working_dir/data","./working_dir/data","./data","../working_dir/data"]
DATA_DIR = None
for p in candidatos_data:
    if Path(p).exists():
        DATA_DIR = Path(p)
        break
if DATA_DIR is None:
    raise RuntimeError("No se encontró la carpeta de datos")

OUT_DIR = Path("/home/gus/work/out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "parquet").mkdir(parents=True, exist_ok=True)

spanish_months = {"enero":1,"febrero":2,"marzo":3,"abril":4,"mayo":5,"junio":6,"julio":7,"agosto":8,"septiembre":9,"setiembre":9,"octubre":10,"noviembre":11,"diciembre":12}
spanish_weekdays = ["lunes","martes","miercoles","miércoles","jueves","viernes","sabado","sábado","domingo"]

def norm_text(s):
    if s is None:
        return None
    return re.sub(r"\s+","_",unidecode(str(s)).strip().lower())

sinonimos = {
    "anio": {"anio","ano","year"},
    "mes": {"mes","month"},
    "departamento": {"departamento","depto"},
    "municipio": {"municipio","muni"},
    "zona": {"zona"},
    "tipo_accidente": {"tipo_accidente","tipo_de_accidente","clase","clase_accidente","clase_acc"},
    "hora": {"hora","hr","hora_del_dia"},
    "dia_semana": {"dia_semana","dia_de_la_semana","dia","día","weekday"},
    "color": {"color"},
    "sexo_conductor": {"sexo_conductor","sexo","genero","genero_conductor","sexo_del_conductor"},
    "condicion_victima": {"condicion","condicion_victima","estado_victima","resultado_victima"},
    "edad": {"edad","age"},
    "tipo_victima": {"tipo_victima","victima_tipo","categoria_victima"}
}

def canon_name(col):
    n = norm_text(col)
    for canon, alts in sinonimos.items():
        if n in alts:
            return canon
    for canon, alts in sinonimos.items():
        for a in alts:
            if a in n:
                return canon
    return n

def normalizar_columnas(df):
    df = df.copy()
    m = {c: canon_name(c) for c in df.columns}
    df.columns = [m[c] for c in df.columns]
    df = df.loc[:, ~pd.Index(df.columns).duplicated(keep='first')]
    return df

def ensure_series(x):
    if isinstance(x, pd.DataFrame):
        return x.iloc[:,0]
    return x

def parse_mes_col(serie):
    s = ensure_series(serie)
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").astype("Int64")
    s = s.astype(str).str.strip().str.lower().map(lambda x: unidecode(x))
    s = s.map(lambda v: spanish_months.get(v, v))
    s = pd.to_numeric(s, errors="coerce").astype("Int64")
    return s

def parse_hora_col(serie):
    s = ensure_series(serie)
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").clip(lower=0, upper=23).astype("Int64")
    s = s.astype(str).str.extract(r"(\d{1,2})", expand=False)
    s = pd.to_numeric(s, errors="coerce").clip(lower=0, upper=23).astype("Int64")
    return s

def agregar_anio_por_filename(df, path):
    if "anio" in df.columns:
        return df
    m = re.search(r"(20\d{2}|19\d{2})", str(path))
    if m:
        df = df.copy()
        df["anio"] = int(m.group(1))
    return df

def cargar_tabla(archivo):
    if archivo.suffix.lower() == ".sav":
        df, meta = pyreadstat.read_sav(str(archivo), apply_value_formats=True)
        return df
    if archivo.suffix.lower() in [".xlsx",".xls"]:
        return pd.read_excel(archivo)
    return None

def limpiar_basica(df):
    df = normalizar_columnas(df)
    if "mes" in df.columns:
        df["mes"] = parse_mes_col(df["mes"])
    if "hora" in df.columns:
        df["hora"] = parse_hora_col(df["hora"])
    if "dia_semana" in df.columns:
        df["dia_semana"] = df["dia_semana"].astype(str).str.strip().str.lower().map(lambda x: unidecode(x))
    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].astype(str).str.strip().str.upper()
    if "municipio" in df.columns:
        df["municipio"] = df["municipio"].astype(str).str.strip().str.title()
    if "zona" in df.columns:
        df["zona"] = df["zona"].astype(str).str.extract(r"(\d+)", expand=False)
    if "tipo_accidente" in df.columns:
        df["tipo_accidente"] = df["tipo_accidente"].astype(str).str.strip().str.title()
    if "color" in df.columns:
        df["color"] = df["color"].astype(str).str.strip().str.title()
    if "sexo_conductor" in df.columns:
        df["sexo_conductor"] = df["sexo_conductor"].astype(str).str.strip().str.title()
    if "condicion_victima" in df.columns:
        df["condicion_victima"] = df["condicion_victima"].astype(str).str.strip().str.title()
    if "edad" in df.columns:
        df["edad"] = pd.to_numeric(df["edad"], errors="coerce").astype("Int64")
    return df

def cargar_unificar(base):
    patrones = [f"{base}_*.sav",f"{base}_*.xlsx"]
    archivos = []
    for pat in patrones:
        archivos.extend(sorted((DATA_DIR).glob(pat)))
    dfs = []
    for a in archivos:
        df = cargar_tabla(a)
        if df is None:
            continue
        df = agregar_anio_por_filename(df, a)
        df = limpiar_basica(df)
        dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    if "anio" in df.columns:
        df = df[df["anio"].between(2014,2023, inclusive="both")]
    return df

def sanitize_parquet(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        s = df[c]
        if s.dtype == object:
            s = s.map(lambda v: v.decode('utf-8', 'ignore') if isinstance(v, (bytes, bytearray)) else v)
            s = s.astype("string")
        elif pd.api.types.is_bool_dtype(s):
            s = s.astype("boolean")
        df[c] = s
    return df

frames_victimas = []
if not df_fallecidos.empty:
    frames_victimas.append(df_fallecidos)
if not df_lesionados.empty:
    frames_victimas.append(df_lesionados)

if frames_victimas:
    cols_union = sorted(set().union(*[f.columns for f in frames_victimas]))
    frames_victimas = [f.reindex(columns=cols_union) for f in frames_victimas]
    df_victimas = pd.concat(frames_victimas, ignore_index=True)
else:
    df_victimas = pd.DataFrame()

df_hechos_pq = sanitize_parquet(df_hechos)
df_vehiculos_pq = sanitize_parquet(df_vehiculos)
df_victimas_pq = sanitize_parquet(df_victimas)

df_hechos_pq.reset_index(drop=True).to_parquet(OUT_DIR / "parquet" / "hechos.parquet", index=False)
df_vehiculos_pq.reset_index(drop=True).to_parquet(OUT_DIR / "parquet" / "vehiculos.parquet", index=False)
df_victimas_pq.reset_index(drop=True).to_parquet(OUT_DIR / "parquet" / "victimas.parquet", index=False)

(len(df_hechos_pq), len(df_vehiculos_pq), len(df_victimas_pq))



(70435, 105665, 100357)

In [17]:
spark = SparkSession.builder.getOrCreate()

OUT_BASE = "./out/parquet"
hechos = spark.read.parquet(f"{OUT_BASE}/hechos.parquet").cache()
vehiculos = spark.read.parquet(f"{OUT_BASE}/vehiculos.parquet").cache()
victimas = spark.read.parquet(f"{OUT_BASE}/victimas.parquet").cache()

try:
    display
except NameError:
    def display(df): df.show(20, truncate=False)

hechos.count(), vehiculos.count(), victimas.count()

(70435, 105665, 100357)

## Ejercicios

In [19]:
## Ejercicio 1

hechos.count(), vehiculos.count(), victimas.count()

hechos.show(10, truncate=False)

vehiculos.show(10, truncate=False)

victimas.show(10, truncate=False)

sel_h = [c for c in ["anio","mes","departamento","municipio","zona","tipo_accidente","hora","dia_semana"] if c in hechos.columns]
hechos.select(sel_h).summary().show(truncate=False)

sel_vh = [c for c in ["anio","mes","departamento","municipio","zona","tipo_accidente","hora","dia_semana","color","sexo_conductor"] if c in vehiculos.columns]
vehiculos.select(sel_vh).summary().show(truncate=False)

sel_v = [c for c in ["anio","mes","departamento","municipio","zona","tipo_accidente","hora","dia_semana","condicion_victima","edad","tipo_victima"] if c in victimas.columns]
victimas.select(sel_v).summary().show(truncate=False)

+---------+----------+---+----+-------------+-------------------------+---------+----+--------------+----+-----------+-----------+--------+----------+---------+------------+----------+------+---------------+----------+------------+----------+--------+---------+------------+
|num_hecho|dia_semana|mes|hora|departamento |mupio_ocu                |areag_ocu|zona|sexo_conductor|edad|mayor_menor|tipo_veh   |color   |modelo_veh|causa_acc|marca_veh   |estado_pil|anio  |num_correlativo|corre_base|area_geo_ocu|estado_con|tipo_eve|num_corre|g_modelo_veh|
+---------+----------+---+----+-------------+-------------------------+---------+----+--------------+----+-----------+-----------+--------+----------+---------+------------+----------+------+---------------+----------+------------+----------+--------+---------+------------+
|NULL     |1.0       |1  |3   |GUATEMALA    |Guatemala                |NULL     |7   |Hombre        |20  |Mayor      |Automóvil  |Negro   |1998.0    |NULL     |Mazda       |NU

In [20]:
## Ejercicio 2

hechos.select("anio").distinct().orderBy("anio").show(100, truncate=False)

vehiculos.select("anio").distinct().orderBy("anio").show(100, truncate=False)

victimas.select("anio").distinct().orderBy("anio").show(100, truncate=False)

+------+
|anio  |
+------+
|2014.0|
|2015.0|
|2016.0|
|2017.0|
|2018.0|
|2019.0|
|2020.0|
|2021.0|
|2022.0|
|2023.0|
+------+

+------+
|anio  |
+------+
|2014.0|
|2015.0|
|2016.0|
|2017.0|
|2018.0|
|2019.0|
|2020.0|
|2021.0|
|2022.0|
|2023.0|
+------+

+------+
|anio  |
+------+
|2014.0|
|2015.0|
|2016.0|
|2017.0|
|2018.0|
|2019.0|
|2020.0|
|2021.0|
|2022.0|
|2023.0|
+------+



In [23]:
## ejercicio 3
def rename_first(df, target, candidates):
    for c in candidates:
        if c in df.columns:
            return df if c == target else df.withColumnRenamed(c, target)
    return df

hechos = rename_first(hechos, "tipo_accidente", ["tipo_accidente","tipo_eve","clase","clase_acc","tipo_evento","tipo"])
hechos = rename_first(hechos, "municipio", ["municipio","mupio_ocu","muni"])
hechos = rename_first(hechos, "departamento", ["departamento","depto"])
hechos = rename_first(hechos, "hora", ["hora","hr","hora_del_dia"])
hechos = rename_first(hechos, "dia_semana", ["dia_semana","dia","día","weekday"])
hechos = rename_first(hechos, "zona", ["zona"])
hechos.cache()


hechos.select("tipo_accidente").distinct().orderBy("tipo_accidente").show(200, truncate=False)


+--------------+
|tipo_accidente|
+--------------+
|1             |
|2             |
|3             |
|4             |
|5             |
|Atropello     |
|Ca?da         |
|Caida         |
|Caída         |
|Choque        |
|Colisi?n      |
|Colision      |
|Colisión      |
|Derrape       |
|Embarranc?    |
|Embarranco    |
|Embarrancó    |
|Encunet?      |
|Encuneto      |
|Ignorado      |
|Vuelco        |
+--------------+



In [24]:
## Ejercicio 4
dep_hechos = hechos.select("departamento").distinct().count() if "departamento" in hechos.columns else None
dep_vehiculos = vehiculos.select("departamento").distinct().count() if "departamento" in vehiculos.columns else None
dep_victimas = victimas.select("departamento").distinct().count() if "departamento" in victimas.columns else None
(dep_hechos, dep_vehiculos, dep_victimas)


(53, 46, 46)

In [25]:
## Ejercicio 5 
hechos_por_anio_dep = hechos.groupBy("anio","departamento").agg(F.count(F.lit(1)).alias("accidentes")).orderBy("anio","departamento")
display(hechos_por_anio_dep)


DataFrame[anio: double, departamento: string, accidentes: bigint]

In [26]:
## Ejercicio 6
acc_por_dia_2023 = hechos.filter(F.col("anio")==2023).groupBy("dia_semana").agg(F.count(F.lit(1)).alias("accidentes")).orderBy(F.desc("accidentes"))
display(acc_por_dia_2023)
acc_por_dia_2023.limit(1).show(truncate=False)

DataFrame[dia_semana: string, accidentes: bigint]

+----------+----------+
|dia_semana|accidentes|
+----------+----------+
|1.0       |324       |
+----------+----------+



In [28]:
## Ejercico 7
dist_hora_muni = hechos.filter(F.col("municipio")==F.lit("Guatemala")).groupBy("hora").agg(F.count(F.lit(1)).alias("accidentes")).orderBy("hora")
display(dist_hora_muni)

DataFrame[hora: bigint, accidentes: bigint]

In [29]:
## Ejercicio 8
llave_hv = [c for c in ["anio","mes","departamento","municipio","zona","tipo_accidente","hora","dia_semana"] if c in hechos.columns and c in vehiculos.columns]
hv_join = hechos.join(vehiculos, on=llave_hv, how="inner")
hv_join.count()

37127